# Brain Tumor Detection and Classification using DenseNet121

Academic workflow for four-class Brain MRI classification.

> **Educational disclaimer:** This notebook is for educational and research purposes only and is not intended to replace professional medical diagnosis.


## 1. Problem Statement

Classify a supplied brain MRI image into one of four dataset categories: **Glioma**, **Meningioma**, **No Tumor**, or **Pituitary Tumor**. The principal architecture is ImageNet-pretrained DenseNet121.


## 2. Import Libraries

This notebook reuses the same project modules as the command-line training pipeline and Streamlit application so class ordering, preprocessing, model construction, evaluation, prediction, and Grad-CAM remain consistent throughout the project.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.data_loader import load_datasets, validate_dataset_structure
from src.eda import run_eda
from src.preprocessing import load_rgb_image, image_to_model_batch, create_data_augmentation
from src.model import build_densenet121_model, find_backbone, prepare_for_fine_tuning
from src.train import train, set_reproducible_seeds
from src.evaluate import evaluate
from src.predict import predict_image, load_trained_model
from app.gradcam import create_heatmap_images

set_reproducible_seeds(RANDOM_SEED)
print("TensorFlow version:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print(EDUCATIONAL_DISCLAIMER)

## 3. Configuration

Important paths and hyperparameters are centralized in `src/config.py`. The same configuration is used by the notebook, training scripts, evaluation pipeline, and Streamlit app.

In [ ]:
config_table = pd.DataFrame({
    "Setting": [
        "Image size", "Input shape", "Batch size", "Number of classes",
        "Validation split", "Initial epochs", "Fine-tune epochs",
        "Initial learning rate", "Fine-tune learning rate",
        "Upper layers considered for fine-tuning", "Random seed"
    ],
    "Value": [
        str(IMAGE_SIZE), str(INPUT_SHAPE), BATCH_SIZE, NUM_CLASSES,
        VALIDATION_SPLIT, INITIAL_EPOCHS, FINE_TUNE_EPOCHS,
        INITIAL_LEARNING_RATE, FINE_TUNE_LEARNING_RATE,
        FINE_TUNE_LAYERS, RANDOM_SEED
    ]
})
display(config_table)
print("Class order:", CLASS_NAMES)
print("Training directory:", TRAINING_DIR)
print("Testing directory:", TESTING_DIR)
print("Model output:", MODEL_PATH)

## 4. Dataset Loading

Expected folder structure is `dataset/Training/<class>/...` and `dataset/Testing/<class>/...`. The project creates an 80/20 training-validation split from the Training folder and reserves the supplied Testing folder for final evaluation.

In [ ]:
validate_dataset_structure()
print("✓ Dataset structure is valid.")

for split_name, split_dir in [("Training", TRAINING_DIR), ("Testing", TESTING_DIR)]:
    print(f"\n{split_name} images:")
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        count = sum(
            1 for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
        )
        print(f"  {DISPLAY_NAMES[class_name]:18s}: {count}")

train_ds, val_ds, test_ds = load_datasets()
train_images, train_labels = next(iter(train_ds))
val_images, val_labels = next(iter(val_ds))
test_images, test_labels = next(iter(test_ds))

print("\nBatch shapes")
print("Training  :", train_images.shape, train_labels.shape)
print("Validation:", val_images.shape, val_labels.shape)
print("Testing   :", test_images.shape, test_labels.shape)

## 5. Exploratory Data Analysis

EDA is calculated from the actual files in the Training folder. It measures image counts, class distribution, observed dimensions, corrupted files, and a simple max-to-min class-count imbalance ratio. No statistics are hard-coded.

In [ ]:
stats = run_eda()

summary_df = pd.DataFrame({
    "Metric": [
        "Total training images",
        "Number of classes",
        "Potential imbalance ratio (max/min)",
        "Corrupted/unreadable files"
    ],
    "Value": [
        stats["total_images"],
        stats["number_of_classes"],
        stats["potential_imbalance_ratio_max_to_min"],
        len(stats["corrupted_files"])
    ]
})
display(summary_df)

class_counts_df = pd.DataFrame({
    "Class": [DISPLAY_NAMES[name] for name in CLASS_NAMES],
    "Images": [stats["images_per_class"][name] for name in CLASS_NAMES]
})
display(class_counts_df)

if stats["image_dimensions"]:
    dimensions_df = pd.DataFrame(
        list(stats["image_dimensions"].items()),
        columns=["Original dimension", "Image count"]
    )
    display(dimensions_df.head(15))

for plot_name in ["class_distribution.png", "sample_mri_grid.png"]:
    plot_path = PLOTS_DIR / plot_name
    if plot_path.exists():
        display(Image.open(plot_path))

if stats["corrupted_files"]:
    print("Corrupted files detected:")
    for item in stats["corrupted_files"][:10]:
        print(" -", item)
else:
    print("✓ No corrupted images detected during EDA.")

## 6. Image Preprocessing

Pipeline: validate image → convert to RGB → resize to 224×224 → convert to float32 → apply DenseNet `preprocess_input`. Grayscale MRI scans are converted to three channels because ImageNet-pretrained DenseNet121 expects RGB input.

In [ ]:
def first_image_in(directory: Path) -> Path:
    for class_name in CLASS_NAMES:
        class_dir = directory / class_name
        for path in sorted(class_dir.iterdir()):
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
                return path
    raise FileNotFoundError(f"No supported image found in {directory}")

demo_image_path = first_image_in(TRAINING_DIR)
with Image.open(demo_image_path) as raw_image:
    raw_copy = raw_image.copy()

processed_image = load_rgb_image(demo_image_path)
model_batch = image_to_model_batch(demo_image_path)

print("Example file:", demo_image_path.relative_to(PROJECT_ROOT))
print("Original mode/size:", raw_copy.mode, raw_copy.size)
print("Processed mode/size:", processed_image.mode, processed_image.size)
print("Model batch shape:", model_batch.shape)
print("Model batch dtype:", model_batch.dtype)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(raw_copy, cmap="gray" if raw_copy.mode != "RGB" else None)
axes[0].set_title("Original MRI")
axes[0].axis("off")
axes[1].imshow(processed_image)
axes[1].set_title("RGB + 224×224")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Data Augmentation

Only training images receive conservative random rotation, zoom, translation, and contrast adjustment. Validation, testing, and single-image inference receive no random augmentation.

In [ ]:
augmentation = create_data_augmentation(RANDOM_SEED)
augmentation.summary()

original_tensor = tf.cast(np.asarray(processed_image), tf.float32)
augmented_versions = [
    augmentation(tf.expand_dims(original_tensor, 0), training=True)[0].numpy()
    for _ in range(6)
]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for axis, image_array, index in zip(axes.ravel(), augmented_versions, range(1, 7)):
    axis.imshow(np.clip(image_array / 255.0, 0, 1))
    axis.set_title(f"Augmented #{index}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 8. DenseNet121 Architecture

DenseNet121 is loaded with ImageNet weights and `include_top=False`. The project adds GlobalAveragePooling2D, BatchNormalization, Dense(256, ReLU, L2), Dropout(0.35), and a 4-unit Softmax output layer.

In [ ]:
model_preview, backbone_preview = build_densenet121_model()

total_params = model_preview.count_params()
trainable_params = sum(np.prod(variable.shape) for variable in model_preview.trainable_weights)
non_trainable_params = total_params - trainable_params

print("Model:", model_preview.name)
print("Backbone:", backbone_preview.name)
print("Backbone frozen initially:", not backbone_preview.trainable)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters initially: {int(trainable_params):,}")
print(f"Non-trainable parameters initially: {int(non_trainable_params):,}")
model_preview.summary()

## 9. Transfer Learning

**Stage 1 — Feature Extraction:** DenseNet121 is frozen and only the custom classification head is trained.

**Stage 2 — Fine-Tuning:** the best Stage-1 checkpoint is reloaded, selected upper DenseNet layers are unfrozen, BatchNormalization layers remain frozen, and training continues with a much smaller learning rate.

In [ ]:
fine_tune_preview = prepare_for_fine_tuning(model_preview, FINE_TUNE_LAYERS)
preview_backbone = find_backbone(fine_tune_preview)
trainable_backbone_layers = [
    layer.name for layer in preview_backbone.layers if layer.trainable
]

print("Trainable DenseNet backbone layers after fine-tuning setup:",
      len(trainable_backbone_layers))
print("Last trainable backbone layers:")
for layer_name in trainable_backbone_layers[-10:]:
    print(" -", layer_name)
print("Fine-tuning learning rate:", FINE_TUNE_LEARNING_RATE)

del model_preview, backbone_preview, fine_tune_preview, preview_backbone
tf.keras.backend.clear_session()

## 10. Model Training

The canonical `train()` pipeline performs EDA, Stage-1 feature extraction, Stage-2 selective fine-tuning, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, history saving, training-curve generation, checkpoint comparison by validation loss, and final model saving.

> Running this cell can take substantial time on CPU.

In [ ]:
# Automatically train only when the final model does not already exist.
# Set RUN_TRAINING = True manually if you intentionally want to retrain.
RUN_TRAINING = not MODEL_PATH.exists()

if RUN_TRAINING:
    trained_model_path = train(
        initial_epochs=INITIAL_EPOCHS,
        fine_tune_epochs=FINE_TUNE_EPOCHS
    )
    print("Training complete:", trained_model_path)
else:
    print("Existing trained model detected:", MODEL_PATH)
    print("Skipping retraining. Set RUN_TRAINING=True to retrain from scratch.")

if TRAINING_HISTORY_PATH.exists():
    history = json.loads(TRAINING_HISTORY_PATH.read_text(encoding="utf-8"))
    print("Stage-1 epochs completed:", history.get("stage1_epochs_completed"))
    print("Stage-2 epochs completed:", history.get("stage2_epochs_completed"))
    for filename in ["training_accuracy.png", "training_loss.png"]:
        path = PLOTS_DIR / filename
        if path.exists():
            display(Image.open(path))

## 11. Fine-Tuning

Fine-tuning is deliberately selective rather than unfreezing the entire pretrained network. Lower layers keep general visual features, while selected upper layers adapt to MRI-specific patterns. A lower learning rate reduces the risk of destroying useful pretrained representations.

In [ ]:
print("Upper DenseNet layers considered for fine-tuning:", FINE_TUNE_LAYERS)
print("Initial learning rate:", INITIAL_LEARNING_RATE)
print("Fine-tuning learning rate:", FINE_TUNE_LEARNING_RATE)
print("Learning-rate reduction factor:", INITIAL_LEARNING_RATE / FINE_TUNE_LEARNING_RATE)
print("Final model exists:", MODEL_PATH.exists())

## 12. Evaluation

Evaluation is performed on the held-out Testing folder. The pipeline calculates actual test accuracy, macro precision, macro recall, macro F1, classification report, confusion matrix, class-wise metrics, and one-vs-rest ROC/AUC where mathematically valid.

In [ ]:
if MODEL_PATH.exists():
    metrics = evaluate()
    metrics_table = pd.DataFrame(
        [(key, value) for key, value in metrics.items() if key != "one_vs_rest_auc"],
        columns=["Metric", "Value"]
    )
    display(metrics_table)

    if "one_vs_rest_auc" in metrics:
        auc_table = pd.DataFrame(
            [(DISPLAY_NAMES[key], value) for key, value in metrics["one_vs_rest_auc"].items()],
            columns=["Class", "One-vs-Rest AUC"]
        )
        display(auc_table)
else:
    print("No trained model found. Run the training section first.")

## 13. Confusion Matrix

Rows represent true MRI categories and columns represent predicted categories. Correct predictions appear on the diagonal; off-diagonal values reveal class-specific confusion.

In [ ]:
cm_path = PLOTS_DIR / "confusion_matrix.png"
if cm_path.exists():
    display(Image.open(cm_path))
else:
    print("Confusion matrix is not available yet. Train and evaluate the model first.")

## 14. Classification Report

Precision measures how often predictions for a class are correct. Recall measures how many true examples of a class are recovered. F1-score balances precision and recall.

In [ ]:
if CLASSIFICATION_REPORT_PATH.exists():
    report = json.loads(CLASSIFICATION_REPORT_PATH.read_text(encoding="utf-8"))
    report_rows = []
    for class_name in CLASS_NAMES:
        values = report[class_name]
        report_rows.append({
            "Class": DISPLAY_NAMES[class_name],
            "Precision": values["precision"],
            "Recall": values["recall"],
            "F1-score": values["f1-score"],
            "Support": int(values["support"])
        })
    display(pd.DataFrame(report_rows))

    class_metrics_path = PLOTS_DIR / "class_wise_metrics.png"
    roc_path = PLOTS_DIR / "roc_curves.png"
    if class_metrics_path.exists():
        display(Image.open(class_metrics_path))
    if roc_path.exists():
        display(Image.open(roc_path))
else:
    print("Classification report is not available yet.")

## 15. Prediction

The following cell automatically selects a real MRI from the Testing folder and runs the exact same preprocessing and inference pipeline used by the application. The reported confidence and probabilities come from the trained model.

In [ ]:
test_sample_path = first_image_in(TESTING_DIR)
print("Selected test MRI:", test_sample_path.relative_to(PROJECT_ROOT))
display(Image.open(test_sample_path))

if MODEL_PATH.exists():
    prediction = predict_image(test_sample_path)
    print("Predicted MRI category:", prediction["predicted_class"])
    print(f"Model confidence: {prediction['confidence'] * 100:.2f}%")

    probabilities_df = pd.DataFrame({
        "Class": list(prediction["probabilities"].keys()),
        "Probability (%)": [value * 100 for value in prediction["probabilities"].values()]
    })
    display(probabilities_df)

    plt.figure(figsize=(8, 4))
    plt.bar(probabilities_df["Class"], probabilities_df["Probability (%)"])
    plt.ylabel("Probability (%)")
    plt.title("Model Probability Distribution")
    plt.ylim(0, 100)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print("No trained model available for prediction.")

## 16. Grad-CAM

Grad-CAM highlights regions that influenced the neural network's selected class by tracing gradients back to convolutional feature maps. It explains model attention and is **not** a clinically validated tumor-localization or segmentation method.

In [ ]:
if MODEL_PATH.exists():
    trained_model = load_trained_model()
    original, heatmap, overlay = create_heatmap_images(test_sample_path, trained_model)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, image_obj, title in zip(
        axes,
        [original, heatmap, overlay],
        ["Original MRI", "Grad-CAM Heatmap", "Heatmap Overlay"]
    ):
        axis.imshow(image_obj)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    print(GRADCAM_DISCLAIMER)
else:
    print("No trained model available for Grad-CAM.")

## 17. Conclusions

This notebook connects dataset validation, real EDA, consistent preprocessing, conservative augmentation, DenseNet121 transfer learning, two-stage training, selective fine-tuning, measured test-set evaluation, real single-image prediction, and Grad-CAM in one reproducible academic workflow.

Any performance claims must come from the evaluation artifacts generated by the actual model run. Good performance on one public dataset does not establish clinical reliability or generalization to other scanners, hospitals, or patient populations.

**Educational disclaimer:** This project is not a substitute for radiologists, doctors, professional MRI interpretation, or clinical diagnosis.

### Viva recap

- **MRI:** imaging technique that uses magnetic fields and radio-frequency signals to create detailed images of internal structures.
- **Image classification:** assigns an input image to one of a fixed set of categories.
- **CNN:** learns spatial image features using convolution filters.
- **DenseNet:** densely connects layers so features can be reused and gradients can flow efficiently.
- **DenseNet121:** a 121-layer DenseNet variant used here as the pretrained feature backbone.
- **Transfer learning:** adapts visual features learned on a large source dataset to a new task.
- **ImageNet weights:** provide broadly useful pretrained visual features instead of starting from random weights.
- **224×224:** the consistent DenseNet-compatible input size used throughout the project.
- **RGB conversion:** provides the three channels expected by the pretrained model.
- **Data augmentation:** creates conservative variations only for training data to improve generalization.
- **Validation set:** guides training decisions without touching the final test set.
- **Overfitting:** good training performance but poor performance on unseen data.
- **Fine-tuning:** updates selected pretrained layers with a small learning rate.
- **Softmax:** converts the four output scores to a probability distribution.
- **Accuracy:** proportion of all predictions that are correct.
- **Precision:** how often predictions for a class are correct.
- **Recall:** how many true examples of a class are recovered.
- **F1-score:** harmonic mean of precision and recall.
- **Confusion matrix:** compares true classes with predicted classes.
- **ROC/AUC:** evaluates class discrimination across thresholds using one-vs-rest analysis.
- **Grad-CAM:** gradient-based visualization of influential image regions, not a diagnosis or tumor segmentation.
- **Inference flow:** validate image → RGB → resize → DenseNet preprocessing → DenseNet121 → Softmax probabilities → predicted category → Grad-CAM explanation.